In [1]:
import os
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve().parent
TEST_RETRIEVAL_ROOT = PROJECT_ROOT / "data" / "retrieval_test"

TEST_METADATA_DB = TEST_RETRIEVAL_ROOT / "metadata.db"
TEST_MINSEARCH_INDEX = TEST_RETRIEVAL_ROOT / "minsearch_index.pkl"
TEST_MINSEARCH_DOCUMENTS = (
    TEST_RETRIEVAL_ROOT / "minsearch_documents.json"
)
TEST_VECTOR_INDEX = TEST_RETRIEVAL_ROOT / "vector_index.npz"
TEST_VECTOR_METADATA = (
    TEST_RETRIEVAL_ROOT / "vector_index_metadata.json"
)

for path in [
    TEST_METADATA_DB,
    TEST_MINSEARCH_INDEX,
    TEST_MINSEARCH_DOCUMENTS,
    TEST_VECTOR_INDEX,
    TEST_VECTOR_METADATA,
]:
    assert path.exists(), f"Missing test artifact: {path}"


# Set these BEFORE importing retrieval.py or rag_service.py.
os.environ["CLINICAL_SYNOPSIS_METADATA_DB"] = str(TEST_METADATA_DB)
os.environ["CLINICAL_SYNOPSIS_RETRIEVAL_OUTPUT_DIR"] = str(
    TEST_RETRIEVAL_ROOT
)

# Remove already imported modules, if any.
for module_name in [
    "retrieval",
    "rag_service",
    "clinical_synopsis.retrieval",
    "clinical_synopsis.rag_service",
]:
    sys.modules.pop(module_name, None)

print("Testing against:", TEST_RETRIEVAL_ROOT)

Testing against: /Users/barbarato/llm-zoomcamp-capstone-project/data/retrieval_test


In [5]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from clinical_synopsis import rag_service as rag

TEST_PATIENT_ID = "f203e11d-5573-1624-69b8-af8436987b3e"

test_question = (
    "Provide a brief overview of this patient's medical background "
    "and current status."
)

test_out = rag.rag_new(
    query=test_question,
    patient_id=TEST_PATIENT_ID,
    question_type="patient_overview",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(f"Patient: {test_out['patient_name']}")
print(f"DOB: {test_out['patient_dob']}")
print(f"Age: {test_out['patient_age_years']}")
print(f"Gender: {test_out['patient_gender']}")

print("\n=== Answer ===")
print(test_out["answer"])

print("\n=== Test context size ===")
print(f"{len(test_out['context']):,} characters")

print("\n=== Generation usage ===")
print(f"Input tokens: {test_out.get('input_tokens', 0):,}")
print(f"Output tokens: {test_out.get('output_tokens', 0):,}")
print(f"Estimated cost: ${test_out.get('total_cost', 0.0):.6f}")

Patient: Shawana711 Lakin515
DOB: 1965-01-05
Age: 61
Gender: female

=== Answer ===
1. **Summary:**
- The documented clinical conditions include active breast cancer diagnosed in 2015 with stage II / stage 2A, HER2-negative, prior resolved acute myeloid leukemia, resolved acute pulmonary embolism, and active hypoxemia; a concussion with loss of consciousness was also documented as resolved.
- The most recent clinically important status is that hypoxemia remains active, while the recent concussion, viral sinusitis, pulmonary embolism, pneumonia, sepsis, and respiratory distress are documented as resolved.
- Oncology timeline shows active breast malignancy identified in May 2015 with T2N0, M0, stage 2/2A disease, followed by a completed radiotherapy course and repeated observations from 2015 to 2019 noting the cancer condition improved.

2. **Active Conditions:**
- **Malignant neoplasm of breast (disorder)** — active; date: 2015-05-07
- **Hypoxemia (disorder)** — active; date: 2020-05-04